# Frequency analysis

The mass matrix is calculated with `Shell.calc_kM` and the eigenvalue
problem $(\boldsymbol{K} + \lambda^2 \boldsymbol{M})\boldsymbol{c} = \boldsymbol{0}$ is solved with
`structsolve.freq`, where $\lambda^2 = -\omega_n^2$ and $\omega_n$ is
the natural frequency in rad/s. The effect of a pre-stress is included by
adding the geometric stiffness matrix to the constitutive stiffness matrix. The
code below is extracted from one of the `panels` unit tests:

In [1]:
import numpy as np
from structsolve import freq
from panels.shell import Shell

In [2]:
# definitions of tests/tests_shell/test_natural_frequencies.py used below
models = ['plate_clpt_donnell', 'cylshell_clpt_donnell',
          'cylshell_clpt_sanders', 'plate_fsdt_donnell', 'plate_tsdt_donnell',
          'cylshell_fsdt_donnell', 'cylshell_fsdt_sanders',
          'cylshell_tsdt_donnell', 'cylshell_tsdt_sanders']

In [3]:
# from tests/tests_shell/test_natural_frequencies.py
def test_panel_freq():
    for model in models:
        for prestress in [True, False]:
            print('Frequency Analysis, prestress={0}, model={1}'.format(
                  prestress, model))
            p = Shell()
            p.model = model
            p.a = 1.
            p.b = 0.5
            p.r = 1.e8
            p.stack = [0, 90, -45, +45]
            p.plyt = 1e-3*0.125
            p.laminaprop = (142.5e9, 8.7e9, 0.28, 5.1e9, 5.1e9, 5.1e9)
            p.rho = 1.3e3
            p.m = 8
            p.n = 8
            p.Nxx = -60.
            p.Nyy = -5.
            k0 = p.calc_kC(silent=True)
            M = p.calc_kM(silent=True)
            if prestress:
                kG0 = p.calc_kG(silent=True)
                k0 += kG0
            eigvals, eigvecs = freq(k0, M, sparse_solver=True, silent=True)
            omegan = np.sqrt(-eigvals)
            if prestress:
                assert np.isclose(omegan[0], 17.6002, rtol=0.001)
            else:
                assert np.isclose(omegan[0], 39.1962, rtol=0.001)

In [4]:
test_panel_freq()
print('test_panel_freq: passed')

Frequency Analysis, prestress=True, model=plate_clpt_donnell
Frequency Analysis, prestress=False, model=plate_clpt_donnell
Frequency Analysis, prestress=True, model=cylshell_clpt_donnell
Frequency Analysis, prestress=False, model=cylshell_clpt_donnell
Frequency Analysis, prestress=True, model=cylshell_clpt_sanders
Frequency Analysis, prestress=False, model=cylshell_clpt_sanders
Frequency Analysis, prestress=True, model=plate_fsdt_donnell
Frequency Analysis, prestress=False, model=plate_fsdt_donnell
Frequency Analysis, prestress=True, model=plate_tsdt_donnell
Frequency Analysis, prestress=False, model=plate_tsdt_donnell


Frequency Analysis, prestress=True, model=cylshell_fsdt_donnell
Frequency Analysis, prestress=False, model=cylshell_fsdt_donnell
Frequency Analysis, prestress=True, model=cylshell_fsdt_sanders
Frequency Analysis, prestress=False, model=cylshell_fsdt_sanders
Frequency Analysis, prestress=True, model=cylshell_tsdt_donnell
Frequency Analysis, prestress=False, model=cylshell_tsdt_donnell


Frequency Analysis, prestress=True, model=cylshell_tsdt_sanders
Frequency Analysis, prestress=False, model=cylshell_tsdt_sanders
test_panel_freq: passed
